# Part 1: Save eval datasets to Kaggle artifacts

In [ ]:
!pip install -U transformers sentencepiece datasets huggingface_hub

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from datasets import load_dataset
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
from transformers import AutoTokenizer

# Secrets
user_secrets = UserSecretsClient()
token = user_secrets.get_secret("HF_TOKEN")
login(token=token)

# Repro
seed = 42
np.random.seed(seed)

# Tokenizer config
ADAPTER_REPO = "legesher/language-decoded-lora-condition-2-ur-5k"
BASE_MODEL = "CohereLabs/tiny-aya-base"
MAX_SEQ_LENGTH = 1024

# Kaggle persistence
KAGGLE_WORKING_DIR = Path("/kaggle/working")
ARTIFACT_DIR = KAGGLE_WORKING_DIR / "eval_unsloth_artifacts"
DATASET_OUTPUT_DIR = ARTIFACT_DIR / "datasets"
DATASET_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

LANGUAGE_CONFIGS = {
    "zh": {
        "xnli": "zh",
        "xcsqa": "X-CSQA-zh",
        "flores": "zho_Hans",
    },
    "es": {
        "xnli": "es",
        "xcsqa": "X-CSQA-es",
        "flores": "spa_Latn",
    },
    "ur": {
        "xnli": "ur",
        "xcsqa": "X-CSQA-ur",
        "flores": "urd_Arab",
    },
}

TEMPLATE_TASKS = {
    "template1": {"xnli", "csqa", "sib200", "belebele"},
    "template2": {"xnli", "csqa", "sib200", "belebele"},
}

SIB200_CATEGORIES = [
    "science/technology",
    "travel",
    "politics",
    "sports",
    "health",
    "entertainment",
    "geography",
]

print("Artifacts will be written to:")
print(f"  Datasets: {DATASET_OUTPUT_DIR}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, token=token)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

print(f"Loaded tokenizer from {BASE_MODEL}")

In [ ]:
def dataset_cache_path(name: str) -> Path:
    return DATASET_OUTPUT_DIR / f"{name}.jsonl"


def load_cached_dataframe(name: str):
    path = dataset_cache_path(name)
    if path.exists():
        print(f"Loading cached {name} from {path}")
        return pd.read_json(path, lines=True)
    return None


def save_cached_dataframe(name: str, df: pd.DataFrame) -> pd.DataFrame:
    path = dataset_cache_path(name)
    df.to_json(path, orient="records", lines=True, force_ascii=False)
    print(f"Saved {name} to {path}")
    return df


def get_cached_dataframe(name: str, builder):
    cached = load_cached_dataframe(name)
    if cached is not None:
        return cached
    df = builder()
    return save_cached_dataframe(name, df)

In [ ]:
import re

LABEL_MAP = {0: "entailment", 1: "neutral", 2: "contradiction"}
ANSWER_LETTERS_4 = {"1": "A", "2": "B", "3": "C", "4": "D", 1: "A", 2: "B", 3: "C", 4: "D"}


def normalize_xnli_gold(label):
    if isinstance(label, str):
        return label.strip().lower()
    return LABEL_MAP.get(int(label), None)


def normalize_csqa_gold(label):
    return str(label).strip().upper()


def normalize_belebele_gold(label):
    return ANSWER_LETTERS_4.get(label, ANSWER_LETTERS_4.get(str(label).strip()))


def normalize_sib200_gold(label):
    return str(label).strip().lower()


def build_xnli_prompt(row, lang: str, template_id: str = "template1") -> str:
    templates = {
        "template1": {
            "en": f"""Read the premise and hypothesis below.
Decide whether the hypothesis is entailed by the premise, contradicted by the premise, or neither.
Reply with only one word: entailment, contradiction, or neutral.
Premise: {row['premise']}
Hypothesis: {row['hypothesis']}
Answer:""",
            "es": f"""Lee la premisa y la hipótesis a continuación.
Decide si la hipótesis se sigue de la premisa, la contradice, o no es ninguna de las dos.
Responde con solo una palabra: entailment, contradiction, o neutral.
Premisa: {row['premise']}
Hipótesis: {row['hypothesis']}
Respuesta:""",
            "zh": f"""请阅读下面的前提和假设。
判断假设是否被前提蕴含、与前提矛盾，或两者都不是。
只回答一个词：entailment、contradiction 或 neutral。
前提: {row['premise']}
假设: {row['hypothesis']}
答案:""",
            "ur": f"""نیچے دی گئی premise اور hypothesis کو پڑھیں۔
فیصلہ کریں کہ hypothesis، premise سے لازم آتی ہے، اس کی تردید کرتی ہے، یا دونوں میں سے کوئی نہیں۔
صرف ایک لفظ میں جواب دیں: entailment، contradiction، یا neutral۔
Premise: {row['premise']}
Hypothesis: {row['hypothesis']}
جواب:""",
        },
        "template2": {
            "en": f"""Compare the premise with the hypothesis.
Choose the relationship: entailment, contradiction, or neutral.
Respond with only that one word.
Premise: {row['premise']}
Hypothesis: {row['hypothesis']}
Answer:""",
            "es": f"""Compara la premisa con la hipótesis.
Elige la relación: entailment, contradiction, o neutral.
Responde solo con esa palabra.
Premisa: {row['premise']}
Hipótesis: {row['hypothesis']}
Respuesta:""",
            "zh": f"""比较前提和假设。
选择二者关系：entailment、contradiction 或 neutral。
只回复这个词。
前提: {row['premise']}
假设: {row['hypothesis']}
答案:""",
            "ur": f"""premise اور hypothesis کا موازنہ کریں۔
رشتہ منتخب کریں: entailment، contradiction، یا neutral۔
صرف وہی ایک لفظ جواب دیں۔
Premise: {row['premise']}
Hypothesis: {row['hypothesis']}
جواب:""",
        },
    }
    return templates[template_id][lang]


def build_csqa_prompt(row, lang: str, template_id: str) -> str:
    templates = {
        "template1": {
            "en": f"""Choose the best answer to the following commonsense question.
Reply with only one letter: A, B, C, D, or E.
Question: {row['stem']}
A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
Answer:""",
            "es": f"""Elige la mejor respuesta para la siguiente pregunta de sentido común.
Responde solo con una letra: A, B, C, D o E.
Pregunta: {row['stem']}
A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
Respuesta:""",
            "zh": f"""请选择下面这个常识问题的最佳答案。
只回答一个字母：A、B、C、D 或 E。
问题: {row['stem']}
A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
答案:""",
            "ur": f"""نیچے دیے گئے عمومی فہم کے سوال کا بہترین جواب منتخب کریں۔
صرف ایک حرف میں جواب دیں: A، B، C، D، یا E۔
سوال: {row['stem']}
A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
جواب:""",
        },
        "template2": {
            "en": f"""Answer this commonsense question by selecting the most suitable option.
Output a single letter only: A, B, C, D, or E.
Question: {row['stem']}
A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
Answer:""",
            "es": f"""Responde esta pregunta de sentido común seleccionando la opción más adecuada.
Escribe solo una letra: A, B, C, D o E.
Pregunta: {row['stem']}
A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
Respuesta:""",
            "zh": f"""请为这个常识问题选择最合适的选项。
只输出一个字母：A、B、C、D 或 E。
问题: {row['stem']}
A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
答案:""",
            "ur": f"""اس عمومی فہم کے سوال کے لیے سب سے مناسب اختیار منتخب کریں۔
صرف ایک حرف لکھیں: A، B، C، D، یا E۔
سوال: {row['stem']}
A. {row['A']}
B. {row['B']}
C. {row['C']}
D. {row['D']}
E. {row['E']}
جواب:""",
        },
    }
    return templates[template_id][lang]


def build_sib200_prompt(row, lang: str, template_id: str) -> str:
    categories = ", ".join(SIB200_CATEGORIES)
    templates = {
        "template1": {
            "en": f"""Classify the following text into one topic category.
Reply with only one category from this list: {categories}.
Text: {row['text']}
Category:""",
            "es": f"""Clasifica el siguiente texto en una categoría temática.
Responde solo con una categoría de esta lista: {categories}.
Texto: {row['text']}
Categoría:""",
            "zh": f"""请将下面的文本分类到一个主题类别中。
只回答此列表中的一个类别：{categories}。
文本: {row['text']}
类别:""",
            "ur": f"""درج ذیل متن کو ایک موضوعی زمرے میں درجہ بند کریں۔
اس فہرست میں سے صرف ایک زمرہ جواب دیں: {categories}۔
متن: {row['text']}
زمرہ:""",
        },
        "template2": {
            "en": f"""Assign one topic label to the text below.
Use exactly one of these labels: {categories}.
Text: {row['text']}
Topic:""",
            "es": f"""Asigna una etiqueta temática al texto siguiente.
Usa exactamente una de estas etiquetas: {categories}.
Texto: {row['text']}
Tema:""",
            "zh": f"""为下面的文本指定一个主题标签。
请只使用这些标签中的一个：{categories}。
文本: {row['text']}
主题:""",
            "ur": f"""نیچے دیے گئے متن کو ایک موضوعی لیبل دیں۔
ان لیبلز میں سے بالکل ایک استعمال کریں: {categories}۔
متن: {row['text']}
موضوع:""",
        },
    }
    return templates[template_id][lang]


def build_belebele_prompt(row, lang: str, template_id: str) -> str:
    templates = {
        "template1": {
            "en": f"""Read the passage and answer the question.
Reply with only one letter: A, B, C, or D.
Passage: {row['flores_passage']}
Question: {row['question']}
A. {row['mc_answer1']}
B. {row['mc_answer2']}
C. {row['mc_answer3']}
D. {row['mc_answer4']}
Answer:""",
            "es": f"""Lee el pasaje y responde la pregunta.
Responde solo con una letra: A, B, C o D.
Pasaje: {row['flores_passage']}
Pregunta: {row['question']}
A. {row['mc_answer1']}
B. {row['mc_answer2']}
C. {row['mc_answer3']}
D. {row['mc_answer4']}
Respuesta:""",
            "zh": f"""阅读文章并回答问题。
只回答一个字母：A、B、C 或 D。
文章: {row['flores_passage']}
问题: {row['question']}
A. {row['mc_answer1']}
B. {row['mc_answer2']}
C. {row['mc_answer3']}
D. {row['mc_answer4']}
答案:""",
            "ur": f"""اقتباس پڑھیں اور سوال کا جواب دیں۔
صرف ایک حرف میں جواب دیں: A، B، C، یا D۔
اقتباس: {row['flores_passage']}
سوال: {row['question']}
A. {row['mc_answer1']}
B. {row['mc_answer2']}
C. {row['mc_answer3']}
D. {row['mc_answer4']}
جواب:""",
        },
        "template2": {
            "en": f"""Use the passage to choose the correct answer to the question.
Respond with just one letter: A, B, C, or D.
Passage: {row['flores_passage']}
Question: {row['question']}
A. {row['mc_answer1']}
B. {row['mc_answer2']}
C. {row['mc_answer3']}
D. {row['mc_answer4']}
Answer:""",
            "es": f"""Usa el pasaje para elegir la respuesta correcta a la pregunta.
Responde solo con una letra: A, B, C o D.
Pasaje: {row['flores_passage']}
Pregunta: {row['question']}
A. {row['mc_answer1']}
B. {row['mc_answer2']}
C. {row['mc_answer3']}
D. {row['mc_answer4']}
Respuesta:""",
            "zh": f"""根据文章选择问题的正确答案。
只回复一个字母：A、B、C 或 D。
文章: {row['flores_passage']}
问题: {row['question']}
A. {row['mc_answer1']}
B. {row['mc_answer2']}
C. {row['mc_answer3']}
D. {row['mc_answer4']}
答案:""",
            "ur": f"""اقتباس کی بنیاد پر سوال کا درست جواب منتخب کریں۔
صرف ایک حرف جواب دیں: A، B، C، یا D۔
اقتباس: {row['flores_passage']}
سوال: {row['question']}
A. {row['mc_answer1']}
B. {row['mc_answer2']}
C. {row['mc_answer3']}
D. {row['mc_answer4']}
جواب:""",
        },
    }
    return templates[template_id][lang]


PROMPT_BUILDERS = {
    "xnli": build_xnli_prompt,
    "csqa": build_csqa_prompt,
    "sib200": build_sib200_prompt,
    "belebele": build_belebele_prompt,
}


def add_precomputed_columns(dataset_name: str, df: pd.DataFrame, tokenizer) -> pd.DataFrame:
    task, dataset_lang = dataset_name.split("_", 1)
    prepared = df.copy()

    if task == "xnli":
        prepared["gold"] = prepared["label"].apply(normalize_xnli_gold)
    elif task == "csqa":
        prepared["gold"] = prepared["answerKey"].apply(normalize_csqa_gold)
    elif task == "sib200":
        prepared["gold"] = prepared["category"].apply(normalize_sib200_gold)
    elif task == "belebele":
        prepared["gold"] = prepared["correct_answer_num"].apply(normalize_belebele_gold)
    else:
        raise ValueError(f"Unsupported dataset for preprocessing: {dataset_name}")

    for template_id, tasks in TEMPLATE_TASKS.items():
        if task not in tasks:
            continue
        for prompt_suffix, prompt_lang in [("english", "en"), ("language", dataset_lang)]:
            prompt_column = f"prompt_{template_id}_{prompt_suffix}"
            input_ids_column = f"input_ids_{template_id}_{prompt_suffix}"
            attention_mask_column = f"attention_mask_{template_id}_{prompt_suffix}"
            prepared[prompt_column] = prepared.apply(
                lambda row, builder=PROMPT_BUILDERS[task], lang=prompt_lang, template=template_id: builder(row, lang, template),
                axis=1,
            )
            encoded = tokenizer(
                prepared[prompt_column].astype(str).tolist(),
                padding=False,
                truncation=True,
                max_length=MAX_SEQ_LENGTH,
            )
            prepared[input_ids_column] = encoded["input_ids"]
            prepared[attention_mask_column] = encoded["attention_mask"]

    return prepared

## Build datasets

In [ ]:
# XNLI
xnli_zh = get_cached_dataframe("xnli_zh", lambda: load_dataset("facebook/xnli", LANGUAGE_CONFIGS["zh"]["xnli"])["test"].to_pandas())
xnli_es = get_cached_dataframe("xnli_es", lambda: load_dataset("facebook/xnli", LANGUAGE_CONFIGS["es"]["xnli"])["test"].to_pandas())
xnli_ur = get_cached_dataframe("xnli_ur", lambda: load_dataset("facebook/xnli", LANGUAGE_CONFIGS["ur"]["xnli"])["test"].to_pandas())

In [ ]:
# X-CSQA
def prepare_csqa(df):
    df = df[["question", "answerKey"]].copy()
    df["stem"] = df["question"].apply(lambda x: x["stem"])
    df["choice_texts"] = df["question"].apply(lambda x: list(x["choices"]["text"]))
    df["A"] = df["choice_texts"].apply(lambda x: x[0])
    df["B"] = df["choice_texts"].apply(lambda x: x[1])
    df["C"] = df["choice_texts"].apply(lambda x: x[2])
    df["D"] = df["choice_texts"].apply(lambda x: x[3])
    df["E"] = df["choice_texts"].apply(lambda x: x[4])
    return df[["stem", "A", "B", "C", "D", "E", "answerKey"]]


def build_csqa(config):
    return prepare_csqa(load_dataset("INK-USC/xcsr", config)["validation"].to_pandas())


csqa_zh = get_cached_dataframe("csqa_zh", lambda: build_csqa(LANGUAGE_CONFIGS["zh"]["xcsqa"]))
csqa_es = get_cached_dataframe("csqa_es", lambda: build_csqa(LANGUAGE_CONFIGS["es"]["xcsqa"]))
csqa_ur = get_cached_dataframe("csqa_ur", lambda: build_csqa(LANGUAGE_CONFIGS["ur"]["xcsqa"]))

In [ ]:
# SIB-200
def build_sib200(flores_code: str):
    df = load_dataset("mteb/sib200", split="test").to_pandas()
    df = df[df["lang"] == flores_code].copy()
    return df[["index_id", "text", "category", "lang"]]


sib200_zh = get_cached_dataframe("sib200_zh", lambda: build_sib200(LANGUAGE_CONFIGS["zh"]["flores"]))
sib200_es = get_cached_dataframe("sib200_es", lambda: build_sib200(LANGUAGE_CONFIGS["es"]["flores"]))
sib200_ur = get_cached_dataframe("sib200_ur", lambda: build_sib200(LANGUAGE_CONFIGS["ur"]["flores"]))

In [ ]:
# Belebele
def build_belebele(flores_code: str):
    columns = [
        "link",
        "question_number",
        "flores_passage",
        "question",
        "mc_answer1",
        "mc_answer2",
        "mc_answer3",
        "mc_answer4",
        "correct_answer_num",
        "dialect",
    ]
    return load_dataset("facebook/belebele", flores_code)["test"].to_pandas()[columns]


belebele_zh = get_cached_dataframe("belebele_zh", lambda: build_belebele(LANGUAGE_CONFIGS["zh"]["flores"]))
belebele_es = get_cached_dataframe("belebele_es", lambda: build_belebele(LANGUAGE_CONFIGS["es"]["flores"]))
belebele_ur = get_cached_dataframe("belebele_ur", lambda: build_belebele(LANGUAGE_CONFIGS["ur"]["flores"]))

## Final artifact summary

In [ ]:
SAVE_DATASETS = {
    "xnli_zh": xnli_zh,
    "xnli_es": xnli_es,
    "xnli_ur": xnli_ur,
    "csqa_zh": csqa_zh,
    "csqa_es": csqa_es,
    "csqa_ur": csqa_ur,
    "sib200_zh": sib200_zh,
    "sib200_es": sib200_es,
    "sib200_ur": sib200_ur,
    "belebele_zh": belebele_zh,
    "belebele_es": belebele_es,
    "belebele_ur": belebele_ur,
}

for dataset_name, dataframe in SAVE_DATASETS.items():
    output_path = DATASET_OUTPUT_DIR / f"{dataset_name}.jsonl"
    prepared_dataframe = add_precomputed_columns(dataset_name, dataframe, tokenizer)
    prepared_dataframe.to_json(output_path, orient="records", lines=True, force_ascii=False)
    print(f"Saved {dataset_name} with prompts, tokens, and normalized gold to {output_path}")

print("\nDone. Commit this notebook, then attach the resulting Kaggle Dataset to notebook 2.")